In [1]:
from retrieval import IndexConfig, FilterConfig, TaxonSearcher,build_index
import tantivy
import json
import os
from collections import OrderedDict
from tantivy import Query
from retrieval import IndexConfig
from retrieval.config import DEFAULT_STATUSES
from retrieval.utils import norm_keyword

In [2]:
cfg = IndexConfig.from_env()

In [3]:
# index = build_index(cfg)

In [4]:
# load index
index = tantivy.Index.open(str(cfg.index_dir))  
searcher = index.searcher() # General search object from tantivy

In [5]:
with open(os.path.join(cfg.index_dir, "meta.json")) as f:
    meta = json.load(f)

fields = [f["name"] for f in meta["schema"]]
print(fields)               # all field names
print('--')
print([f for f in meta["schema"] if f["name"] in {"taxon_rank","taxonomic_status"}])

['taxon_id', 'species_id', 'canonical_name', 'scientific_name', 'generic_name', 'vernaculars', 'name_text', 'canonical_edge', 'scientific_edge', 'generic_edge', 'vernacular_edge', 'taxon_rank', 'taxonomic_status', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'source', 'marine', 'freshwater', 'terrestrial', 'extinct', 'hybrid', 'age_days', 'size_mm', 'mass_g', 'curated_description', 'scientific_name_authorship', 'lifeform_curated', 'habitat_curated', 'livingperiod_curated', 'nomenclatural_status']
--
[{'name': 'taxon_rank', 'type': 'text', 'options': {'indexing': {'record': 'position', 'fieldnorms': True, 'tokenizer': 'raw'}, 'stored': True, 'fast': True}}, {'name': 'taxonomic_status', 'type': 'text', 'options': {'indexing': {'record': 'position', 'fieldnorms': True, 'tokenizer': 'raw'}, 'stored': True, 'fast': True}}]


In [6]:
total = searcher.num_docs
total

7694321

In [7]:
searcher.num_segments

9

In [8]:
agg = searcher.aggregate(
    Query.all_query(),
    {"by_status": {"terms": {"field": "taxonomic_status", "size": 2000}}},
)
print(agg["by_status"]["buckets"])


[{'key': 'accepted', 'doc_count': 4124085}, {'key': 'synonym', 'doc_count': 2912885}, {'key': 'doubtful', 'doc_count': 298719}, {'key': 'homotypic_synonym', 'doc_count': 189270}, {'key': 'heterotypic_synonym', 'doc_count': 150913}, {'key': 'proparte_synonym', 'doc_count': 18448}]


In [9]:
agg = searcher.aggregate(
    Query.all_query(),
    {"by_status": {"terms": {"field": "taxon_rank", "size": 2000}}},
)
print(agg["by_status"]["buckets"])


[{'key': 'species', 'doc_count': 4961031}, {'key': 'unranked', 'doc_count': 1266880}, {'key': 'genus', 'doc_count': 544187}, {'key': 'variety', 'doc_count': 418975}, {'key': 'subspecies', 'doc_count': 377671}, {'key': 'form', 'doc_count': 87374}, {'key': 'family', 'doc_count': 33907}, {'key': 'order', 'doc_count': 3087}, {'key': 'class', 'doc_count': 843}, {'key': 'phylum', 'doc_count': 347}, {'key': 'kingdom', 'doc_count': 19}]


In [10]:
ts = TaxonSearcher(index_dir=str(cfg.index_dir)) 

In [34]:
text = """
Game Vertebrate Densities in Hunted and Nonhunted Forest Sites in Manu National Park, Peru	Manu National Park of southern Peru is one of the most renowned protected areas in the world, yet large-bodied vertebrate surveys conducted to date have been restricted to Cocha Cashu Biological Station, a research station covering <0.06 percent of the 1.7Mha park. Manu Park is occupied by >460 settled Matsigenka Amerindians, 300-400 isolated Matsigenka, and several, little-known groups of isolated hunter-gatherers, yet the impact of these native Amazonians on game vertebrate populations within the park remains poorly understood. On the basis of 1495 km of standardized line-transect censuses, we present density and biomass estimates for 23 mammal, bird, and reptile species for seven lowland and upland forest sites in Manu Park, including Cocha Cashu. We compare these estimates between hunted and nonhunted sites within Manu Park, and with other Neotropical forest sites. Manu Park safeguards some of the most species-rich and highest biomass assemblages of arboreal and terrestrial mammals ever recorded in Neotropical forests, most likely because of its direct Andean influence and high levels of soil fertility. Relative to Barro Colorado Island, seed predators and arboreal folivores in Manu are rare, and generalist frugivores specializing on mature fruit pulp are abundant. The impact of such a qualitative shift in the vertebrate community on the dynamics of plant regeneration, and therefore, on our understanding of tropical plant ecology, must be profound. Despite a number of external threats, Manu Park continues to serve as a baseline against which other Neotropical forests can be gauged.
"""

out = ts.search(
    text,
    limit=10
)

In [35]:
print(f"RAW: {len(out.raw_candidates)}", out.raw_candidates[:10])
print(f"KEPT: {len(out.kept_candidates)}", out.kept_candidates[:10])
print(f"HIR: {len(out.hits)}")

match= []
for r in out.hits[:]:
    # print(r["score"], r["taxon_id"], r["taxon_rank"], r["scientific_name"], r["taxonomic_status"]) 
    _json= {
    "canonical_name": r["canonical_name"],
    "scientific_name": r["scientific_name"],
    "vernaculars": r["vernaculars"],
    "generic_name": r["generic_name"],
    "taxon_rank":r["taxon_rank"],
    "taxonomic_status": r["taxonomic_status"],

    "related_kingdom" : r["kingdom"], 
    "related_phylum" : r['phylum'], 
    "related_class" : r['class'], 
    "related_order" : r['order'], 
    "related_family" : r['family'], 
    "related_genus" : r['genus'],
    "description": r["curated_description"],
    "scientific_name_authorship": r["scientific_name_authorship"],
    "score": r["score"]    


}
    match.append(_json)

RAW: 136 ['Park of southern', 'Park of', 'Peru is one', 'Peru is', 'Park is occupied', 'Park is', 'Neotropical forest sites', 'Neotropical forest', 'Park safeguards some', 'Park safeguards']
KEPT: 37 ['Park of southern', 'Park of', 'Peru is one', 'Peru is', 'Park is', 'Neotropical forest', 'Park safeguards', 'Neotropical forests', 'Andean influence', 'Relative to']
HIR: 10


In [36]:
match

[{'canonical_name': 'Manu',
  'scientific_name': 'Manu Marples, 1946',
  'vernaculars': None,
  'generic_name': 'Manu',
  'taxon_rank': 'genus',
  'taxonomic_status': 'accepted',
  'related_kingdom': 'animalia',
  'related_phylum': 'chordata',
  'related_class': 'aves',
  'related_order': None,
  'related_family': None,
  'related_genus': 'manu',
  'description': None,
  'scientific_name_authorship': 'Marples, 1946',
  'score': 200.3739776611328},
 {'canonical_name': 'Peru',
  'scientific_name': 'Peru Koçak & Kemal, 2008',
  'vernaculars': None,
  'generic_name': 'Peru',
  'taxon_rank': 'genus',
  'taxonomic_status': 'accepted',
  'related_kingdom': 'animalia',
  'related_phylum': 'arthropoda',
  'related_class': 'insecta',
  'related_order': 'orthoptera',
  'related_family': 'phalangopsidae',
  'related_genus': 'peru',
  'description': 'habitat: Terrestrial.',
  'scientific_name_authorship': 'Koçak & Kemal, 2008',
  'score': 190.4346160888672},
 {'canonical_name': 'Manu antiquus',
  '

In [22]:
{
    "canonical_name": r["canonical_name"],
    "scientific_name": r["scientific_name"],
    "vernaculars": r["vernaculars"],
    "generic_name": r["generic_name"],
    "taxon_rank":r["taxon_rank"],
    "taxonomic_status": r["taxonomic_status"],

    "related_kingdom" : r["kingdom"], 
    "related_phylum" : r['phylum'], 
    "related_class" : r['class'], 
    "related_order" : r['order'], 
    "related_family" : r['family'], 
    "related_genus" : r['genus'],
    "description": r["curated_description"],
    "scientific_name_authorship": r["scientific_name_authorship"],
    "score": r["score"]    


}

{'canonical_name': 'Boletus viscidocorrugis',
 'scientific_name': 'Boletus viscidocorrugis Both',
 'vernaculars': None,
 'generic_name': 'Boletus',
 'taxon_rank': 'species',
 'taxonomic_status': 'accepted',
 'related_kingdom': 'fungi',
 'related_phylum': 'basidiomycota',
 'related_class': 'agaricomycetes',
 'related_order': 'boletales',
 'related_family': 'boletaceae',
 'related_genus': 'boletus',
 'description': None,
 'scientific_name_authorship': 'Both',
 'score': 77.7920150756836}

In [17]:
out.hits[0]

{'score': 209.31439208984375,
 'taxon_id': 2012126,
 'species_id': None,
 'canonical_name': 'Bemisia tabaci',
 'scientific_name': 'Bemisia tabaci (Gennadius, 1889)',
 'generic_name': 'Bemisia',
 'vernaculars': 'Sweetpotato whitefly (en)',
 'taxon_rank': 'species',
 'taxonomic_status': 'accepted',
 'kingdom': 'animalia',
 'phylum': 'arthropoda',
 'class': 'insecta',
 'order': 'hemiptera',
 'family': 'aleyrodidae',
 'genus': 'bemisia',
 'source': None,
 'marine': 0,
 'freshwater': 0,
 'terrestrial': 0,
 'extinct': 0,
 'hybrid': 0,
 'age_days': None,
 'size_mm': None,
 'mass_g': None,
 'curated_description': 'eunis habitat: Arable land and market gardens (I1 level 2)\nnative range: Asia-Temperate\nnative range: Asia-Tropical\nnative range: Cosmopolitan, Cryptogenic\neunis habitat: Greenhouse (added 21oct2005) (J100 level 2)\necofunctional group: Phytophagous\nBemisia tabaci (Gennadius) Gennadius: Bemisia tabaci (Gennadius) (Figs 15–24) B. tabaci was originally described by Gennadius (1889